# ДЗ 3. NumPy на данных «Дала Маркет»
Срок: **до четверга 12.11, 19:00**.
Сдача — папка `hw03` репозитория `dsa-homework` (`hw03.ipynb`): `git add hw03` → `git commit` → `git push`.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем — только массивы и маски (кроме задачи 8).

- **[База]** обязательно для всех · **[Стандарт]** — сколько получится · **[Челлендж]** — для тех, кому легко.

## [Для всех] Чтение кода — без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
# 1
a = np.arange(6).reshape(2, 3); print(a[1, 2], a.shape)
# 2
a = np.array([3, 8, 1, 8]); print(a.argmax(), a.max())
# 3
a = np.arange(10); b = a[::3]; b[0] = 99; print(a[0])
# 4
a = np.array([1, 5, 7, 2]); print((a > 3).sum(), (a > 3).mean())
# 5
print(np.zeros((2, 3)).shape, np.linspace(0, 1, 5)[1])
```

Мои предсказания и объяснения:

1. будет 5 и при запросе shape он запросит размер и ответит 2,3
2. ответ будет 1, 8. 1 это индекс максимального числа а 8 это максимальный элемент
3. 99
4. 12 и 3
5. 0 0 0
6. 0 0 0

7. выйдут  5 элементов от 0 до 0.99

In [3]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [5]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

# функции из урока 2 + превращаем колонки в массивы NumPy
amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])

print(amount.shape, amount.dtype, paid.dtype)
print(city)

(5463,) float64 bool
['Алматы' 'Астана' 'Караганда' ... 'Астана' 'Алматы' 'Алматы']


## [База]
1. Найдите позицию (`argmax`) самого дорогого заказа. Какая сумма, клиент и город?
2. Выручка оплаченных заказов и сумма неоплаченных — через маску `paid` и `~paid`.
3. Сколько заказов в Шымкенте и какая у них доля от всех заказов?

In [8]:
#1
num = amount.argmax()
print(amount.argmax(), amount.max(), cust[num], city[num])

1001 9999990.0 TEST Алматы


In [10]:
#2
print("Сумма оплаченых заказов: ", amount[paid].sum())
print("Сумма неоплаченых заказов: ",amount[~paid].sum())

Сумма оплаченых заказов:  101909360.0
Сумма неоплаченых заказов:  7974000.0


In [12]:
#3
shym = len(amount[city == "Шымкент"])
print("Всего заказов в Шымкенте: ",shym)
print("Доля Шымкента от всех заказов составляет: ", (shym/len(amount))*100)

Всего заказов в Шымкенте:  804
Доля Шымкента от всех заказов составляет:  14.71718835804503


## [Стандарт]
4. Маркетинг предлагает скидку 15% на оплаченные заказы дороже 50 000 ₸. Сколько выручки магазин недополучит?
   Посчитайте дважды: со всеми заказами и без заказов клиента TEST. Почему ответы различаются в несколько раз?
5. Для **каждого** города — доля заказов, доставленных за 1–2 дня. Список городов: `np.unique(city)`; цикл по 6 городам здесь можно.
6. Ниже код, который AI написал на запрос «посчитай средний чек». Найдите две ошибки, исправьте и посчитайте правильно:
```python
avg = amount.mean()
print("Средний чек:", round(avg))
```

In [15]:
# Стандарт
#1
mask = (paid) & (amount>50000)
notest = cust != "TEST"
masknotest = mask & notest
print("Сумма оплаченных заказов больше 50000: ", amount[mask].sum())
print("Сумма оплаченных заказов больше 50000: ", amount[masknotest].sum())

Сумма оплаченных заказов больше 50000:  38016150.0
Сумма оплаченных заказов больше 50000:  8016180.0


In [28]:
for c in np.unique(city):
    mask_city = city == c
    mask_delivery = (days >= 1) & (days <= 2)

    total_orders = mask_city.sum()
    fast_orders = (mask_city & mask_delivery).sum()

    share = (fast_orders / total_orders) * 100

    print(c, share)

Актобе 62.21662468513854
Алматы 64.53382084095064
Астана 63.52517985611511
Атырау 64.38356164383562
Караганда 67.84386617100373
Шымкент 29.850746268656714


In [30]:
avg = amount.mean()
print("Средний чек:", avg)
#здесь сложно что либо сказать так как не понятна сама задача, если посчитать весь средний чек всех городов то все правильно

Средний чек: 20114.105802672526


## [Челлендж]
7. Сравните скорость: сумма оплаченных заказов циклом `for` и через маску. Во сколько раз быстрее?
8. Смоделируйте A/B-тест. `np.random.seed(42)`. Группа A: 1 000 клиентов, вероятность покупки 5%; группа B: 1 000 клиентов, 6%
   (покупка = `np.random.rand(1000) < 0.05`). Сколько покупок в каждой группе?
   Повторите эксперимент 100 раз: в скольких случаях B оказалась лучше A? Что это говорит о размере выборки? (2–3 предложения)

In [ ]:
# Челлендж